# Bonus - SGD vs Momentum vs AdaGrad vs RMSProp vs Adam with the Rosenbrock function


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation
from matplotlib.patches import FancyArrowPatch
import matplotlib as mpl

torch.manual_seed(42)
np.random.seed(42)


In [ ]:
class Optimizer:
    def __init__(self, lr):
        self.lr = lr

    def init_state(self, theta):
        return None

    def step(self, theta, grad, state, step_idx):
        raise NotImplementedError

    def optimize(self, f, theta0, n_steps=100):
        theta = torch.tensor(theta0, dtype=torch.float32, requires_grad=True)
        state = self.init_state(theta)

        trajectory = [theta.detach().clone()]
        values = [f(theta).item()]

        for step_idx in range(1, n_steps + 1):
            loss = f(theta)
            loss.backward()
            grad = theta.grad.detach().clone()

            with torch.no_grad():
                theta, state = self.step(theta, grad, state, step_idx)

            theta.grad.zero_()
            trajectory.append(theta.detach().clone())
            values.append(f(theta).item())

        return torch.stack(trajectory), values

class GradientDescentOptimizer(Optimizer):
    def step(self, theta, grad, state, step_idx):
        theta -= self.lr * grad
        return theta, state

class MomentumOptimizer(Optimizer):
    def __init__(self, lr, beta=0.9):
        super().__init__(lr)
        self.beta = beta

    def init_state(self, theta):
        return {"velocity": torch.zeros_like(theta)}

    def step(self, theta, grad, state, step_idx):
        state["velocity"] = self.beta * state["velocity"] + grad
        theta -= self.lr * state["velocity"]
        return theta, state

class AdaGradOptimizer(Optimizer):
    def __init__(self, lr, eps=1e-8):
        super().__init__(lr)
        self.eps = eps

    def init_state(self, theta):
        return {"grad_sq_sum": torch.zeros_like(theta)}

    def step(self, theta, grad, state, step_idx):
        state["grad_sq_sum"] += grad ** 2
        theta -= self.lr * grad / (torch.sqrt(state["grad_sq_sum"]) + self.eps)
        return theta, state

class RMSPropOptimizer(Optimizer):
    def __init__(self, lr, beta=0.9, eps=1e-8):
        super().__init__(lr)
        self.beta = beta
        self.eps = eps

    def init_state(self, theta):
        return {"avg_sq_grad": torch.zeros_like(theta)}

    def step(self, theta, grad, state, step_idx):
        state["avg_sq_grad"] = self.beta * state["avg_sq_grad"] + (1 - self.beta) * (grad ** 2)
        theta -= self.lr * grad / (torch.sqrt(state["avg_sq_grad"]) + self.eps)
        return theta, state

class AdamOptimizer(Optimizer):
    def __init__(self, lr, beta1=0.9, beta2=0.999, eps=1e-8):
        super().__init__(lr)
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps

    def init_state(self, theta):
        return {
            "m": torch.zeros_like(theta),
            "v": torch.zeros_like(theta),
        }

    def step(self, theta, grad, state, step_idx):
        state["m"] = self.beta1 * state["m"] + (1 - self.beta1) * grad
        state["v"] = self.beta2 * state["v"] + (1 - self.beta2) * (grad ** 2)

        m_hat = state["m"] / (1 - self.beta1 ** step_idx)
        v_hat = state["v"] / (1 - self.beta2 ** step_idx)

        theta -= self.lr * m_hat / (torch.sqrt(v_hat) + self.eps)
        return theta, state

def gradient_descent(f, theta0, lr=0.001, n_steps=200):
    optimizer = GradientDescentOptimizer(lr=lr)
    return optimizer.optimize(f, theta0, n_steps=n_steps)

def momentum(f, theta0, lr=0.001, beta=0.9, n_steps=200):
    optimizer = MomentumOptimizer(lr=lr, beta=beta)
    return optimizer.optimize(f, theta0, n_steps=n_steps)

def adagrad(f, theta0, lr=0.1, eps=1e-8, n_steps=200):
    optimizer = AdaGradOptimizer(lr=lr, eps=eps)
    return optimizer.optimize(f, theta0, n_steps=n_steps)

def rmsprop(f, theta0, lr=0.01, beta=0.9, eps=1e-8, n_steps=200):
    optimizer = RMSPropOptimizer(lr=lr, beta=beta, eps=eps)
    return optimizer.optimize(f, theta0, n_steps=n_steps)

def adam(f, theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, n_steps=200):
    optimizer = AdamOptimizer(lr=lr, beta1=beta1, beta2=beta2, eps=eps)
    return optimizer.optimize(f, theta0, n_steps=n_steps)

In [ ]:
def plot_trajectories_contour(f, results, xlim=(-3, 3), ylim=(-2, 2), title="Optimization Trajectories"):
    x_values = np.linspace(xlim[0], xlim[1], 400)
    y_values = np.linspace(ylim[0], ylim[1], 400)
    X, Y = np.meshgrid(x_values, y_values)

    grid = np.stack((X, Y), axis=-1)
    Z = f(torch.tensor(grid, dtype=torch.float32)).detach().numpy()

    plt.figure(figsize=(8, 6))
    plt.contour(X, Y, np.log1p(np.abs(Z)), levels=30)

    for name, (trajectory, _) in results.items():
        traj = trajectory.detach().numpy()
        plt.plot(traj[:, 0], traj[:, 1], marker='o', markersize=2, label=name)

    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_values(results, title):
    plt.figure(figsize=(9, 5))
    for name, (_, values) in results.items():
        plt.plot(values, label=name)
    plt.xlabel("Iteration")
    plt.ylabel("Function value")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation
from matplotlib.patches import FancyArrowPatch
import matplotlib as mpl

mpl.rcParams["animation.embed_limit"] = 80

def rosenbrock(theta, a=1.0, b=100.0):
    x, y = theta[0], theta[1]
    return (a - x) ** 2 + b * (y - x ** 2) ** 2

def build_step_metrics(f, results):
    metrics = {}
    for name, (trajectory, values) in results.items():
        traj_np = trajectory.detach().cpu().numpy()
        deltas = np.diff(traj_np, axis=0)
        magnitudes = np.linalg.norm(deltas, axis=1)
        downhill = []
        for point in traj_np[:-1]:
            theta = torch.tensor(point, dtype=torch.float32, requires_grad=True)
            loss = f(theta)
            loss.backward()
            grad = theta.grad.detach().cpu().numpy()
            downhill.append(-grad)
        downhill = np.array(downhill)
        downhill_norm = np.linalg.norm(downhill, axis=1)
        metrics[name] = {
            "trajectory": traj_np,
            "values": np.array(values),
            "deltas": deltas,
            "delta_norms": magnitudes,
            "downhill": downhill,
            "downhill_norms": downhill_norm,
            "abs_dx": np.abs(deltas[:, 0]),
            "abs_dy": np.abs(deltas[:, 1]),
        }
    return metrics

def animate_rosenbrock_trajectories(f, metrics, xlim=(-2, 2), ylim=(-1, 3), title="Trajectories — Rosenbrock"):
    x = np.linspace(*xlim, 200)
    y = np.linspace(*ylim, 200)
    X, Y = np.meshgrid(x, y)
    Z = f(torch.tensor([X, Y], dtype=torch.float32)).numpy()
    fig, ax = plt.subplots(figsize=(7, 6))
    contours = ax.contour(X, Y, Z, levels=np.logspace(-1, 3, 16), cmap="viridis")
    ax.clabel(contours, inline=True, fontsize=8, fmt="%.1f")
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

    lines = {}
    points = {}
    downhill_arrows = {}
    update_arrows = {}
    legend_downhill = ax.plot([], [], color="tab:blue", linewidth=2, label="downhill direction -∇f")[0]
    legend_update = ax.plot([], [], color="magenta", linewidth=2, label="optimizer update Δw")[0]
    for name in metrics:
        line, = ax.plot([], [], linewidth=2, label=name)
        point, = ax.plot([], [], marker="o", markersize=6)
        lines[name] = line
        points[name] = point
        downhill_arrows[name] = ax.quiver([], [], [], [], angles="xy", scale_units="xy", scale=1, color="tab:blue", alpha=0.7)
        update_arrows[name] = ax.quiver([], [], [], [], angles="xy", scale_units="xy", scale=1, color="magenta", alpha=0.7)
    ax.legend()

    max_frames = max(len(data["trajectory"]) for data in metrics.values())
    downhill_samples = np.concatenate([data["downhill_norms"] for data in metrics.values()])
    delta_samples = np.concatenate([data["delta_norms"] for data in metrics.values()])
    reference_norm = max(
        np.percentile(downhill_samples, 95),
        np.percentile(delta_samples, 95),
        1e-12,
    )
    min_visible_length = 0.05
    max_visible_length = 0.42

    def update(frame):
        for name, data in metrics.items():
            traj = data["trajectory"]
            idx = min(frame + 1, len(traj))
            lines[name].set_data(traj[:idx, 0], traj[:idx, 1])
            points[name].set_data([traj[idx - 1, 0]], [traj[idx - 1, 1]])
            points[name].set_color(lines[name].get_color())
            if idx - 1 < len(data["deltas"]):
                point = traj[idx - 1]
                downhill_norm = max(data["downhill_norms"][idx - 1], 1e-12)
                delta_norm = max(data["delta_norms"][idx - 1], 1e-12)
                downhill_ratio = min(downhill_norm / reference_norm, 1.0)
                delta_ratio = min(delta_norm / reference_norm, 1.0)
                downhill_length = min_visible_length + (max_visible_length - min_visible_length) * np.sqrt(downhill_ratio)
                actual_length = min_visible_length + (max_visible_length - min_visible_length) * np.sqrt(delta_ratio)
                downhill = data["downhill"][idx - 1] / downhill_norm * downhill_length
                actual = data["deltas"][idx - 1] / delta_norm * actual_length
                downhill_arrows[name].set_offsets(point[None, :])
                downhill_arrows[name].set_UVC(np.array([downhill[0]]), np.array([downhill[1]]))
                update_arrows[name].set_offsets(point[None, :])
                update_arrows[name].set_UVC(np.array([actual[0]]), np.array([actual[1]]))
            else:
                downhill_arrows[name].set_offsets(np.empty((0, 2)))
                downhill_arrows[name].set_UVC(np.array([]), np.array([]))
                update_arrows[name].set_offsets(np.empty((0, 2)))
                update_arrows[name].set_UVC(np.array([]), np.array([]))
        return [legend_downhill, legend_update] + list(lines.values()) + list(points.values()) + list(downhill_arrows.values()) + list(update_arrows.values())

    anim = FuncAnimation(fig, update, frames=max_frames, interval=80, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())

def animate_absolute_updates(metrics, key, title, ylabel):
    fig, ax = plt.subplots(figsize=(9, 5))
    max_value = max(np.max(data[key]) for data in metrics.values())
    ax.set_xlim(0, max(len(data[key]) for data in metrics.values()) - 1)
    ax.set_ylim(0, max_value * 1.05 if max_value > 0 else 1)
    ax.set_xlabel("Iteration")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

    lines = {}
    for name in metrics:
        line, = ax.plot([], [], linewidth=2, label=name)
        lines[name] = line
    ax.legend()

    max_frames = max(len(data[key]) for data in metrics.values())

    def update(frame):
        for name, data in metrics.items():
            series = data[key]
            idx = min(frame + 1, len(series))
            lines[name].set_data(np.arange(idx), series[:idx])
        return list(lines.values())

    anim = FuncAnimation(fig, update, frames=max_frames, interval=80, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())

def animate_compass_panel(metrics, title="Compass Panel — Rosenbrock"):
    fig, ax = plt.subplots(figsize=(8, 8))
    anchors = {
        "SGD": (-1.0, 1.0),
        "Momentum": (1.0, 1.0),
        "AdaGrad": (0.0, 0.0),
        "RMSProp": (-1.0, -1.0),
        "Adam": (1.0, -1.0),
    }
    colors = {
        "SGD": "tab:gray",
        "Momentum": "tab:orange",
        "AdaGrad": "tab:green",
        "RMSProp": "tab:olive",
        "Adam": "tab:red",
    }
    ax.set_xlim(-1.8, 1.8)
    ax.set_ylim(-1.8, 1.8)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Local x direction")
    ax.set_ylabel("Local y direction")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

    downhill_arrows = {}
    update_arrows = {}
    labels = {}
    delta_text = {}
    legend_downhill = ax.plot([], [], color="tab:blue", linewidth=2, label="gradient f / downhill -∇f")[0]
    legend_update = ax.plot([], [], color="magenta", linewidth=2, label="optimizer delta w")[0]
    for name, anchor in anchors.items():
        ax.plot(anchor[0], anchor[1], marker="o", color=colors[name], markersize=6)
        labels[name] = ax.text(anchor[0], anchor[1] + 0.18, name, ha="center", va="bottom", fontsize=11, fontweight="bold", color=colors[name])
        delta_text[name] = ax.text(anchor[0], anchor[1] - 0.24, "", ha="center", va="top", fontsize=9)
        downhill_arrows[name] = FancyArrowPatch(anchor, anchor, arrowstyle='-|>', mutation_scale=16, linewidth=2.2, color='tab:blue')
        update_arrows[name] = FancyArrowPatch(anchor, anchor, arrowstyle='-|>', mutation_scale=16, linewidth=2.2, color='magenta')
        ax.add_patch(downhill_arrows[name])
        ax.add_patch(update_arrows[name])
    ax.legend(handles=[legend_downhill, legend_update], loc="lower center")

    max_frames = max(len(data["deltas"]) for data in metrics.values())
    downhill_samples = np.concatenate([data["downhill_norms"] for data in metrics.values()])
    delta_samples = np.concatenate([data["delta_norms"] for data in metrics.values()])
    reference_norm = max(
        np.percentile(downhill_samples, 95),
        np.percentile(delta_samples, 95),
        1e-12,
    )
    min_visible_length = 0.7
    max_visible_length = 0.78

    def update(frame):
        for name, anchor in anchors.items():
            data = metrics[name]
            idx = min(frame, len(data["deltas"]) - 1)
            downhill_norm = max(data["downhill_norms"][idx], 1e-12)
            delta_norm = max(data["delta_norms"][idx], 1e-12)
            downhill_ratio = min(downhill_norm / reference_norm, 1.0)
            delta_ratio = min(delta_norm / reference_norm, 1.0)
            downhill_length = min_visible_length + (max_visible_length - min_visible_length) * np.sqrt(downhill_ratio)
            actual_length = min_visible_length + (max_visible_length - min_visible_length) * np.sqrt(delta_ratio)
            downhill = data["downhill"][idx] / downhill_norm * downhill_length
            actual = data["deltas"][idx] / delta_norm * actual_length
            downhill_end = (anchor[0] + downhill[0], anchor[1] + downhill[1])
            actual_end = (anchor[0] + actual[0], anchor[1] + actual[1])
            downhill_arrows[name].set_positions(anchor, downhill_end)
            update_arrows[name].set_positions(anchor, actual_end)
            delta_text[name].set_text(
                f"|∇f| = {data['downhill_norms'][idx]:.2e}\n|Δw| = {data['delta_norms'][idx]:.2e}"
            )
        return [legend_downhill, legend_update, *downhill_arrows.values(), *update_arrows.values(), *labels.values(), *delta_text.values()]

    anim = FuncAnimation(fig, update, frames=max_frames, interval=80, blit=False)
    plt.close(fig)
    return HTML(anim.to_jshtml())

theta0_rosenbrock = [-0.5, 2.0]
n_steps_rosenbrock = 100

results_rosenbrock = {
    "SGD": gradient_descent(rosenbrock, theta0_rosenbrock, lr=0.0020, n_steps=n_steps_rosenbrock),
    "Momentum": momentum(rosenbrock, theta0_rosenbrock, lr=0.0010, beta=0.95, n_steps=n_steps_rosenbrock),
    "AdaGrad": adagrad(rosenbrock, theta0_rosenbrock, lr=20.0, n_steps=n_steps_rosenbrock),
    "RMSProp": rmsprop(rosenbrock, theta0_rosenbrock, lr=0.2, beta=0.8, n_steps=n_steps_rosenbrock),
    "Adam": adam(rosenbrock, theta0_rosenbrock, lr=0.05, beta1=0.9, beta2=0.9, n_steps=n_steps_rosenbrock),
}

rosenbrock_metrics = build_step_metrics(rosenbrock, results_rosenbrock)

rosenbrock_summary = {
    name: {
        "final_point": np.round(data["trajectory"][-1], 4).tolist(),
        "final_value": float(np.round(data["values"][-1], 6)),
    }
    for name, data in rosenbrock_metrics.items()
}
rosenbrock_summary


display(animate_rosenbrock_trajectories(rosenbrock, rosenbrock_metrics, xlim=(-2, 2), ylim=(-1, 3), title="Trajectories — Rosenbrock"))
display(animate_compass_panel(rosenbrock_metrics, title="Compass Panel — Rosenbrock"))
display(animate_absolute_updates(rosenbrock_metrics, "abs_dx", title="Absolute Update in X Coordinate — Rosenbrock", ylabel="|Δx|"))
display(animate_absolute_updates(rosenbrock_metrics, "abs_dy", title="Absolute Update in Y Coordinate — Rosenbrock", ylabel="|Δy|"))